**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

**Load Project Utilities & Logging**

In [0]:
%run /Workspace/airline_etl_pipeline/2_utilities
%run /Workspace/airline_etl_pipeline/3_pipeline_log

In [0]:
print(catalog, bronze_schema, silver_schema, gold_schema)
print("Landing:", landing_path)

**Define table names**

In [0]:
bronze_table   = f"{catalog}.{bronze_schema}.flights_raw"
silver_fact    = f"{catalog}.{silver_schema}.fact_flights"
silver_carrier = f"{catalog}.{silver_schema}.dim_carrier"
silver_airport = f"{catalog}.{silver_schema}.dim_airport"
gold_fact      = f"{catalog}.{gold_schema}.fact_flight_delays"
gold_agg       = f"{catalog}.{gold_schema}.agg_delay_summary"

print("Bronze :", bronze_table)
print("Silver :", silver_fact)
print("Gold   :", gold_fact)

**Initialize logging**

In [0]:
init_log_table()
run_id     = str(uuid.uuid4())
start_time = datetime.utcnow()

## Bronze

Read all daily CSVs from S3 landing zone — `s3://bts-air/landing/flights/`

In [0]:
# Read all CSVs from all date folders in S3 landing zone
df = (
    spark.read
    .options(header=True, inferSchema=True)
    .csv(f"{landing_path}*/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total Rows: ", df.count())
df.show(5)

In [0]:
# Write to Bronze Delta table
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

print(f"✅ Bronze written → {bronze_table}")

### Move processed files from landing/ to processed/

Same pattern as your FMCG project — prevents files being re-ingested next run.

In [0]:
date_folders = dbutils.fs.ls(landing_path)
moved = 0
for folder in date_folders:
    try:
        files = dbutils.fs.ls(folder.path)
        for file_info in files:
            dest = file_info.path.replace("landing/flights/", "processed/flights/")
            dbutils.fs.mv(file_info.path, dest, True)
            moved += 1
    except:
        pass

print(f"✅ Moved {moved} files from landing/ → processed/")

## Silver

### Data Quality Cleaning

In [0]:
df_flights = spark.sql(f"SELECT * FROM {bronze_table}")
print("Rows from Bronze:", df_flights.count())
df_flights.show(2)

In [0]:
# 1. Remove rows missing key columns
df_flights = df_flights.filter(
    F.col("FlightDate").isNotNull() &
    F.col("Origin").isNotNull() &
    F.col("Dest").isNotNull()
)

# 2. Cast and guard delay columns
df_flights = df_flights \
    .withColumn("DepDelay",
        F.when(F.col("DepDelay").cast("double") < -120, F.lit(None))
         .otherwise(F.col("DepDelay").cast("double"))) \
    .withColumn("ArrDelay", F.col("ArrDelay").cast("double"))

# 3. Fill delay cause nulls with 0.0
for col_name in ["CarrierDelay", "WeatherDelay", "NASDelay",
                 "SecurityDelay", "LateAircraftDelay"]:
    df_flights = df_flights.withColumn(
        col_name,
        F.coalesce(F.col(col_name).cast("double"), F.lit(0.0))
    )

# 4. CancellationCode — N if not cancelled
df_flights = df_flights.withColumn(
    "CancellationCode",
    F.when(F.col("Cancelled") == 1, F.col("CancellationCode"))
     .otherwise(F.lit("N"))
)

# 5. Parse FlightDate to proper date type
df_flights = df_flights.withColumn(
    "FlightDate", F.to_date("FlightDate", "yyyy-MM-dd")
)

# 6. Delay bucket classification
df_flights = df_flights.withColumn(
    "delay_bucket",
    F.when(F.col("ArrDelay") <= 0,   F.lit("on_time"))
     .when(F.col("ArrDelay") <= 15,  F.lit("minor"))
     .when(F.col("ArrDelay") <= 60,  F.lit("moderate"))
     .otherwise(F.lit("severe"))
)

# 7. Primary delay cause
df_flights = df_flights.withColumn(
    "primary_delay_cause",
    F.when(F.col("WeatherDelay") > 0,      F.lit("weather"))
     .when(F.col("CarrierDelay") > 0,      F.lit("carrier"))
     .when(F.col("NASDelay") > 0,          F.lit("nas"))
     .when(F.col("LateAircraftDelay") > 0, F.lit("late_aircraft"))
     .otherwise(F.lit("none"))
)

# 8. SHA surrogate key
df_flights = df_flights.withColumn(
    "flight_key",
    F.sha2(
        F.concat_ws("_",
            F.col("FlightDate"),
            F.col("Reporting_Airline"),
            F.col("Flight_Number_Reporting_Airline"),
            F.col("Origin"),
            F.col("Dest")
        ), 256
    )
)

# 9. Drop duplicates on surrogate key
df_flights = df_flights.dropDuplicates(["flight_key"])

print("Clean rows:", df_flights.count())
df_flights.show(2)

### Dimension Tables

In [0]:
# dim_carrier
dim_carrier = df_flights.select(
    F.col("Reporting_Airline").alias("carrier_code")
).distinct()

if not spark.catalog.tableExists(silver_carrier):
    dim_carrier.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite").saveAsTable(silver_carrier)
else:
    DeltaTable.forName(spark, silver_carrier).alias("t") \
        .merge(dim_carrier.alias("s"), "t.carrier_code = s.carrier_code") \
        .whenNotMatchedInsertAll().execute()

print(f"✅ dim_carrier → {silver_carrier}")
spark.sql(f"SELECT * FROM {silver_carrier}").show()

In [0]:
# dim_airport
dim_airport = df_flights.select(
    F.col("Origin").alias("airport_code"),
    F.col("OriginCityName").alias("city_name"),
    F.col("OriginState").alias("state")
).union(
    df_flights.select(
        F.col("Dest").alias("airport_code"),
        F.col("DestCityName").alias("city_name"),
        F.col("DestState").alias("state")
    )
).distinct()

if not spark.catalog.tableExists(silver_airport):
    dim_airport.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite").saveAsTable(silver_airport)
else:
    DeltaTable.forName(spark, silver_airport).alias("t") \
        .merge(dim_airport.alias("s"), "t.airport_code = s.airport_code") \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"✅ dim_airport → {silver_airport}")
spark.sql(f"SELECT * FROM {silver_airport} LIMIT 5").show()

### Silver Fact Table — upsert

In [0]:
if not spark.catalog.tableExists(silver_fact):
    df_flights.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite").saveAsTable(silver_fact)
    print(f"✅ Silver fact created → {silver_fact}")
else:
    DeltaTable.forName(spark, silver_fact).alias("silver") \
        .merge(
            df_flights.alias("bronze"),
            "silver.flight_key = bronze.flight_key"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print(f"✅ Silver fact upserted → {silver_fact}")

## Gold

### Fact table with window function

In [0]:
df_silver = spark.sql(f"SELECT * FROM {silver_fact}")

# Window function — latest delay pattern per route
w = Window.partitionBy("Origin", "Dest", "Reporting_Airline") \
          .orderBy(F.col("FlightDate").desc())

df_gold = df_silver \
    .withColumn("rn", F.row_number().over(w)) \
    .withColumn("is_latest_record", (F.col("rn") == 1).cast("int")) \
    .drop("rn")

print("Gold rows:", df_gold.count())
df_gold.show(2)

In [0]:
if not spark.catalog.tableExists(gold_fact):
    df_gold.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite").saveAsTable(gold_fact)
    print(f"✅ Gold fact created → {gold_fact}")
else:
    DeltaTable.forName(spark, gold_fact).alias("gold") \
        .merge(
            df_gold.alias("silver"),
            "gold.flight_key = silver.flight_key"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print(f"✅ Gold fact upserted → {gold_fact}")

### Aggregation table

In [0]:
df_agg = df_gold.groupBy(
    "FlightDate", "Reporting_Airline",
    "Origin", "Dest",
    "delay_bucket", "primary_delay_cause"
).agg(
    F.count("*").alias("total_flights"),
    F.avg("ArrDelay").alias("avg_arr_delay"),
    F.avg("DepDelay").alias("avg_dep_delay"),
    F.max("ArrDelay").alias("max_arr_delay"),
    F.sum(F.when(F.col("Cancelled") == 1, 1).otherwise(0)).alias("cancellations"),
    F.sum(F.when(F.col("Diverted") == 1, 1).otherwise(0)).alias("diversions")
)

if not spark.catalog.tableExists(gold_agg):
    df_agg.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite").saveAsTable(gold_agg)
    print(f"✅ Gold agg created → {gold_agg}")
else:
    DeltaTable.forName(spark, gold_agg).alias("agg") \
        .merge(
            df_agg.alias("new"),
            """agg.FlightDate        = new.FlightDate
               AND agg.Reporting_Airline = new.Reporting_Airline
               AND agg.Origin            = new.Origin
               AND agg.Dest              = new.Dest
               AND agg.delay_bucket      = new.delay_bucket"""
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print(f"✅ Gold agg upserted → {gold_agg}")

display(df_agg.limit(5))

**Write pipeline log**

In [0]:
write_log(
    job_name     = "full_load_fact",
    layer        = "bronze→silver→gold",
    run_id       = run_id,
    start_time   = start_time,
    rows_read    = df_flights.count(),
    rows_written = df_gold.count(),
    status       = "success"
)

**Verify row counts across all layers**

In [0]:
%sql
SELECT 'bronze'      AS layer, COUNT(*) AS rows FROM airline.bronze.flights_raw
UNION ALL
SELECT 'silver_fact' AS layer, COUNT(*) AS rows FROM airline.silver.fact_flights
UNION ALL
SELECT 'dim_carrier' AS layer, COUNT(*) AS rows FROM airline.silver.dim_carrier
UNION ALL
SELECT 'dim_airport' AS layer, COUNT(*) AS rows FROM airline.silver.dim_airport
UNION ALL
SELECT 'gold_fact'   AS layer, COUNT(*) AS rows FROM airline.gold.fact_flight_delays
UNION ALL
SELECT 'gold_agg'    AS layer, COUNT(*) AS rows FROM airline.gold.agg_delay_summary
UNION ALL
SELECT 'logs'        AS layer, COUNT(*) AS rows FROM airline.gold.pipeline_logs;